# Секция 1. Назначение работы: Polars-версия

В этой лабораторной работе мы строим учебное хранилище данных для задачи кредитного скоринга. Важно: здесь не строится модель машинного обучения и не выполняется подготовка признаков для ML. Мы делаем только первые три пункта задания: описываем физическую модель, создаем хранилище и показываем, как Polars может обновлять parquet-данные.

Решение специально написано линейно: без классов, без пакетной архитектуры, без командной строки и без оркестраторов. Такой код проще объяснить на защите: каждая функция делает один понятный шаг.

# Секция 2. Инициализация Polars и путей

Polars – это быстрый DataFrame-движок для локальной аналитической обработки. В этой копии ноутбука все чтение, вертикализация, merge и проверки выполняются на Polars без Spark-сессии.

`DATA_DIR` – папка с исходными parquet-источниками. Функция `find_data_dir()` ищет ее в нескольких типичных местах. Если рядом есть `source.zip`, функция распакует архив и найдет внутри директорию с тремя источниками.

`WAREHOUSE_DIR` – отдельная папка, куда будут записаны таблицы учебного хранилища. Старые исходные файлы при этом не изменяются.

In [10]:
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime
from pathlib import Path


def ensure_polars():
    """Подключает Polars; если пакета нет, устанавливает его через uv."""
    try:
        import polars as pl
        return pl
    except ImportError:
        uv_path = shutil.which("uv")
        if uv_path is None:
            raise RuntimeError("Polars не установлен. Установите его командой: uv add polars")
        subprocess.check_call([uv_path, "pip", "install", "polars"])
        import polars as pl
        return pl


pl = ensure_polars()

SOURCE_NAMES = [
    "client_cards_daily",
    "credit_cards_info",
    "deb_cards_info",
]


def _contains_source_dirs(path):
    path = Path(path)
    return path.exists() and all((path / name).is_dir() for name in SOURCE_NAMES)


def _candidate_base_dirs():
    """Возвращает рабочую директорию, ее родителей и стандартные внешние корни."""
    base_dirs = []
    for root in [Path.cwd(), *Path.cwd().parents, Path("/content"), Path("/mnt/data")]:
        if root.exists() and root not in base_dirs:
            base_dirs.append(root)
    return base_dirs


def find_data_dir():
    """Находит директорию, внутри которой лежат три исходника с parquet-партициями."""
    candidates = [Path("/data")]
    for base_dir in _candidate_base_dirs():
        candidates.extend(
            [
                base_dir / "data",
                base_dir / "source" / "sources",
                base_dir / "source",
            ]
        )

    for candidate in candidates:
        if _contains_source_dirs(candidate):
            return candidate.resolve()

    zip_candidates = [
        Path("source.zip"),
        Path("/content/source.zip"),
        Path("/mnt/data/source.zip"),
    ]
    for zip_path in zip_candidates:
        if zip_path.exists():
            extract_to = zip_path.parent
            print(f"Директории с parquet не найдены. Распаковываю {zip_path} в {extract_to}")
            with zipfile.ZipFile(zip_path, "r") as archive:
                archive.extractall(extract_to)
            break

    for root in _candidate_base_dirs():
        for path in root.rglob("*"):
            if path.is_dir() and _contains_source_dirs(path):
                return path.resolve()

    raise FileNotFoundError(
        "Не удалось найти DATA_DIR. Ожидалась директория с подпапками: "
        + ", ".join(SOURCE_NAMES)
    )


DATA_DIR = find_data_dir()
WAREHOUSE_DIR = "warehouse_polars"

print(f"Polars version = {pl.__version__}")
print(f"DATA_DIR = {DATA_DIR}")
print(f"WAREHOUSE_DIR = {Path(WAREHOUSE_DIR).resolve()}")


Polars version = 1.40.1
DATA_DIR = /Users/avtereshchenko/Desktop/Магистратура/Айрапетян_БД_2_сем/dbscoring/source/sources
WAREHOUSE_DIR = /Users/avtereshchenko/Desktop/Магистратура/Айрапетян_БД_2_сем/dbscoring/notebooks/warehouse_polars


# Секция 3. Описание исходных источников

В задании есть три источника.

`deb_cards_info` и `credit_cards_info` обновляются раз в месяц и хранятся как SCD1. В SCD1 нам важно текущее значение атрибута за ключ, а при повторной загрузке дубль не должен появиться.

`client_cards_daily` обновляется раз в день и хранится как SCD2. В SCD2 история хранится через период действия строки: `row_actual_from` и `row_actual_to`. В этой учебной работе мы не усложняем SCD2, потому что эти поля уже есть в источнике. Мы просто переносим их в вертикальную таблицу.

Также здесь задан список бизнес-атрибутов. Технические поля вроде `row_update_dtime`, `loading_id`, `row_hash_val`, `report_dt`, `row_actual_from`, `row_actual_to` не попадают в `dim_attributes` как бизнес-атрибуты.

In [11]:
# Описываем все исходные системы в одном словаре, чтобы загрузчики не держали параметры в разных местах.
SOURCE_CONFIG = {
    "deb_cards_info": {
        # source_id задает числовой первичный ключ источника для связей с dim_attributes, load_log и витринами.
        "source_id": 1,
        # source_name хранит техническое имя директории источника и человекочитаемое имя в справочнике.
        "source_name": "deb_cards_info",
        # source_description попадет в dim_sources и объяснит бизнес-смысл источника.
        "source_description": "Данные о транзакционной активности клиента по дебетовым картам за отчетный месяц.",
        # update_frequency фиксирует частоту обновления источника: месяц или день.
        "update_frequency": "1 месяц",
        # storage_type показывает, какая логика хранения применяется к источнику: SCD1 или SCD2.
        "storage_type": "SCD1",
        # partition_col указывает имя поля, по которому физически разложены parquet-партиции источника.
        "partition_col": "report_dt",
        # initial_partition задает первую партицию для демонстрации начальной загрузки.
        "initial_partition": "2023-02-28",
        # update_partition задает партицию, которая будет загружаться на шаге обновления.
        "update_partition": "2023-03-31",
        # target_table связывает источник с одной из двух EAV-витрин хранилища.
        "target_table": "client_monthly_attrs_scd1",
        # business_columns перечисляет исходные поля, которые участвуют в вертикализации; client_id остается ключом.
        "business_columns": [
            "client_id",
            "onl_bank_active_1m_nflag",
            "auto_pay_active_qty",
            "cl_income_1m_amt",
            "dep_acc_1st_open_dt",
            "wdr_cash_6m_amt",
            "cash_op_6m_amt",
            "cash_3m_qty",
            "lst_balance_amt",
            "card_active_1m_nflag",
        ],
    },
    "credit_cards_info": {
        # source_id задает числовой первичный ключ источника для связей с dim_attributes, load_log и витринами.
        "source_id": 2,
        # source_name хранит техническое имя директории источника и человекочитаемое имя в справочнике.
        "source_name": "credit_cards_info",
        # source_description попадет в dim_sources и объяснит бизнес-смысл источника.
        "source_description": "Данные по кредитным картам клиента за отчетный месяц.",
        # update_frequency фиксирует частоту обновления источника: месяц или день.
        "update_frequency": "1 месяц",
        # storage_type показывает, какая логика хранения применяется к источнику: SCD1 или SCD2.
        "storage_type": "SCD1",
        # partition_col указывает имя поля, по которому физически разложены parquet-партиции источника.
        "partition_col": "report_dt",
        # initial_partition задает первую партицию для демонстрации начальной загрузки.
        "initial_partition": "2023-02-28",
        # update_partition задает партицию, которая будет загружаться на шаге обновления.
        "update_partition": "2023-03-31",
        # target_table связывает источник с одной из двух EAV-витрин хранилища.
        "target_table": "client_monthly_attrs_scd1",
        # business_columns перечисляет исходные поля, которые участвуют в вертикализации; client_id остается ключом.
        "business_columns": [
            "client_id",
            "client_income_amt",
            "oi_total_amt",
            "act_pl_os_rub_amt",
            "payroll_client_nflag",
            "inf_payroll_rub_amt",
            "legal_entity_amt",
            "otf_loan_rub_amt",
            "otf_fee_rub_amt",
            "inf_transfer_rub_amt",
            "cc_ever_nflag",
        ],
    },
    "client_cards_daily": {
        # source_id задает числовой первичный ключ источника для связей с dim_attributes, load_log и витринами.
        "source_id": 3,
        # source_name хранит техническое имя директории источника и человекочитаемое имя в справочнике.
        "source_name": "client_cards_daily",
        # source_description попадет в dim_sources и объяснит бизнес-смысл источника.
        "source_description": "Данные по клиенту, обновляемые раз в день.",
        # update_frequency фиксирует частоту обновления источника: месяц или день.
        "update_frequency": "1 день",
        # storage_type показывает, какая логика хранения применяется к источнику: SCD1 или SCD2.
        "storage_type": "SCD2",
        # partition_col указывает имя поля, по которому физически разложены parquet-партиции источника.
        "partition_col": "row_actual_to",
        # initial_partition задает первую партицию для демонстрации начальной загрузки.
        "initial_partition": "2023-04-03",
        # update_partition задает партицию, которая будет загружаться на шаге обновления.
        "update_partition": "9999-12-31",
        # target_table связывает источник с одной из двух EAV-витрин хранилища.
        "target_table": "client_daily_attrs_scd2",
        # business_columns перечисляет исходные поля, которые участвуют в вертикализации; client_id остается ключом.
        "business_columns": [
            "client_id",
            "srv_mb_nflag",
            "cc_stoplist",
            "lne_tot_debt_int_ovrd_rub_amt",
            "lne_tot_debt_ovrd_rub_amt",
        ],
    },
}

# Технические поля не попадают в dim_attributes как бизнес-атрибуты.
# Перечисляем технические поля источников, которые нельзя превращать в бизнес-атрибуты dim_attributes.
TECHNICAL_COLUMNS = {
    "row_update_dtime",
    "loading_id",
    "row_hash_val",
    "report_dt",
    "row_actual_from",
    "row_actual_to",
}

# В реальных parquet из архива есть несколько расхождений с формулировкой задания:
# id вместо client_id и опечатка nfalg вместо nflag.
# Задаем точечные переименования реальных parquet-полей в имена из учебной схемы.
COLUMN_ALIASES = {
    "id": "client_id",
    "onl_bank_active_1m_nfalg": "onl_bank_active_1m_nflag",
}

# Храним человекочитаемые описания атрибутов, чтобы dim_attributes была не только техническим справочником.
ATTRIBUTE_DESCRIPTIONS = {
    "onl_bank_active_1m_nflag": "Флаг активности клиента в онлайн-банкинге за 1 месяц.",
    "auto_pay_active_qty": "Количество активных шаблонов Автоплатеж за месяц.",
    "cl_income_1m_amt": "Сумма дохода по клиенту за отчетный месяц по дебетовым картам.",
    "dep_acc_1st_open_dt": "Дата открытия первого депозитного договора клиента.",
    "wdr_cash_6m_amt": "Сумма снятия наличных в рублях за последние 6 месяцев.",
    "cash_op_6m_amt": "Объем операций в рублях за последние 6 месяцев.",
    "cash_3m_qty": "Количество операций за последние 3 месяца.",
    "lst_balance_amt": "Баланс последней действующей дебетовой карты.",
    "card_active_1m_nflag": "Флаг активности по дебетовым картам за 1 месяц.",
    "client_income_amt": "Доход клиента по данным от рисков.",
    "oi_total_amt": "Сумма общего дохода по клиенту за отчетный месяц по всем кредитам.",
    "act_pl_os_rub_amt": "Совокупная задолженность по кредитам наличными.",
    "payroll_client_nflag": "Флаг зарплатного клиента.",
    "inf_payroll_rub_amt": "Сумма зарплатных начислений по всем счетам клиента за месяц в рублях.",
    "legal_entity_amt": "Сумма поступлений от юридических лиц.",
    "otf_loan_rub_amt": "Общая сумма списания в погашение кредита по всем картам клиента за месяц в рублях.",
    "otf_fee_rub_amt": "Общая сумма списания комиссий по всем картам клиента за месяц в рублях.",
    "inf_transfer_rub_amt": "Общая сумма поступлений переводом с карты или счета за месяц в рублях.",
    "cc_ever_nflag": "Флаг наличия кредитной карты когда-либо.",
    "srv_mb_nflag": "Флаг, что у клиента подключен мобильный банк.",
    "cc_stoplist": "Флаг, что клиенту нельзя предлагать кредит.",
    "lne_tot_debt_int_ovrd_rub_amt": "Сумма просрочки по процентам по кредитам.",
    "lne_tot_debt_ovrd_rub_amt": "Сумма просрочки по основному долгу по кредитам.",
}

# Выделяем месячные источники в отдельный список, потому что они пишутся в SCD1-витрину.
MONTHLY_SOURCES = ["deb_cards_info", "credit_cards_info"]
# Отдельно фиксируем daily-источник, потому что он пишется в SCD2-витрину с периодом действия строк.
DAILY_SOURCE = "client_cards_daily"


# Секция 4. Создание схем 5 таблиц хранилища

Физическая модель состоит из пяти таблиц.

`dim_sources` – справочник источников. Поля `valid_from` и `valid_to` позволяют хранить период действия описания источника.

`dim_attributes` – справочник бизнес-атрибутов. Здесь каждому атрибуту назначается простой последовательный `attribute_id`.

`load_log` – журнал загрузок. Он нужен, чтобы понимать, какие партиции уже успешно загружались.

`client_monthly_attrs_scd1` – вертикальная таблица месячных атрибутов. Ключ: `client_id + attribute_id + report_dt`.

`client_daily_attrs_scd2` – вертикальная таблица дневных атрибутов. Ключ: `client_id + attribute_id + row_actual_from`.

`client_id` остается отдельным ключевым полем и не превращается в `attribute_id`.

In [12]:
DIM_SOURCES_SCHEMA = {
    "source_id": pl.Int32,
    "source_name": pl.String,
    "source_description": pl.String,
    "update_frequency": pl.String,
    "row_create_dtime": pl.Datetime,
    "valid_to": pl.String,
    "valid_from": pl.String,
    "row_update_dtime": pl.Datetime,
}

DIM_ATTRIBUTES_SCHEMA = {
    "attribute_id": pl.Int32,
    "attribute_name": pl.String,
    "attribute_description": pl.String,
    "data_type": pl.String,
    "source_id": pl.Int32,
    "update_frequency": pl.String,
    "row_create_dtime": pl.Datetime,
    "row_update_dtime": pl.Datetime,
}

LOAD_LOG_SCHEMA = {
    "load_id": pl.Int64,
    "source_id": pl.Int32,
    "source_report_dt": pl.String,
    "load_start_dtime": pl.Datetime,
    "load_end_dtime": pl.Datetime,
    "target_table": pl.String,
    "load_status": pl.String,
    "loading_id": pl.Int64,
    "error_message": pl.String,
}

CLIENT_MONTHLY_SCHEMA = {
    "client_id": pl.String,
    "attribute_id": pl.Int32,
    "report_dt": pl.String,
    "attribute_value": pl.String,
    "source_id": pl.Int32,
    "row_update_dtime": pl.Datetime,
    "loading_id": pl.Int64,
    "row_hash_val": pl.String,
}

CLIENT_DAILY_SCHEMA = {
    "client_id": pl.String,
    "attribute_id": pl.Int32,
    "attribute_value": pl.String,
    "row_actual_from": pl.String,
    "row_actual_to": pl.String,
    "source_id": pl.Int32,
    "row_update_dtime": pl.Datetime,
    "loading_id": pl.Int64,
    "row_hash_val": pl.String,
}

TABLE_SCHEMAS = {
    "dim_sources": DIM_SOURCES_SCHEMA,
    "dim_attributes": DIM_ATTRIBUTES_SCHEMA,
    "load_log": LOAD_LOG_SCHEMA,
    "client_monthly_attrs_scd1": CLIENT_MONTHLY_SCHEMA,
    "client_daily_attrs_scd2": CLIENT_DAILY_SCHEMA,
}

TABLE_NAMES = list(TABLE_SCHEMAS.keys())


# Секция 5. Вспомогательные функции

Здесь собраны простые функции верхнего уровня. Они нужны, чтобы основной сценарий читался как последовательность учебных шагов: создать Polars-хранилище, выполнить первую загрузку, выполнить обновление и проверить результат.

Что такое вертикализация: исходные таблицы имеют широкий вид, где много бизнес-колонок лежат в одной строке клиента. Например: `client_id, income, balance`. После вертикализации каждая бизнес-колонка становится отдельной строкой: `client_id, attribute_id, attribute_value`. Это удобно, когда список атрибутов нужно хранить единообразно.

Почему `attribute_value` хранится как строка: в одной вертикальной колонке оказываются значения разных типов – числа, даты, флаги. Поэтому для учебной модели проще привести их к `string`.

Зачем нужен `load_log`: без журнала загрузок мы могли бы повторно загрузить ту же самую партицию и получить дубли. Журнал фиксирует успешные загрузки по связке `source_id + source_report_dt + target_table`.

In [13]:
def empty_table(table_name):
    """Создает пустой Polars DataFrame с контрактом нужной таблицы."""
    return pl.DataFrame(schema=TABLE_SCHEMAS[table_name])


def table_path(table_name):
    return Path(WAREHOUSE_DIR) / table_name


def parquet_files(path):
    return sorted(Path(path).glob("*.parquet"))


def read_table_if_exists(table_name):
    """Читает parquet-таблицу хранилища или возвращает пустой DataFrame."""
    path = table_path(table_name)
    files = parquet_files(path) if path.exists() else []
    if files:
        return pl.read_parquet(files).select(TABLE_SCHEMAS[table_name].keys())
    return empty_table(table_name)


def write_table(df, table_name):
    """Перезаписывает одну parquet-таблицу через временную директорию."""
    target = table_path(table_name)
    temp = Path(WAREHOUSE_DIR) / f"__tmp_{table_name}"
    Path(WAREHOUSE_DIR).mkdir(parents=True, exist_ok=True)

    if temp.exists():
        shutil.rmtree(temp)
    temp.mkdir(parents=True, exist_ok=True)

    df.select(TABLE_SCHEMAS[table_name].keys()).write_parquet(temp / "part-00000.parquet")

    if target.exists():
        shutil.rmtree(target)
    shutil.move(str(temp), str(target))


def normalize_source_columns(df):
    """Приводит реальные имена полей parquet к именам из задания."""
    rename_map = {
        old_name: new_name
        for old_name, new_name in COLUMN_ALIASES.items()
        if old_name in df.columns and new_name not in df.columns
    }
    if rename_map:
        df = df.rename(rename_map)

    casts = []
    if "client_id" in df.columns:
        casts.append(pl.col("client_id").cast(pl.String))
    if "loading_id" in df.columns:
        casts.append(pl.col("loading_id").cast(pl.Int64))
    if "report_dt" in df.columns:
        casts.append(pl.col("report_dt").cast(pl.String))
    if "row_actual_from" in df.columns:
        casts.append(pl.col("row_actual_from").cast(pl.String))
    if "row_actual_to" in df.columns:
        casts.append(pl.col("row_actual_to").cast(pl.String))

    return df.with_columns(casts) if casts else df


def read_source_partition(source_name, partition_col, partition_value):
    """Читает parquet из директории нужной партиции целиком."""
    source_root = Path(DATA_DIR) / source_name
    candidates = [
        source_root / f"{partition_col}='{partition_value}'",
        source_root / f"{partition_col}={partition_value}",
    ]

    partition_path = next((candidate for candidate in candidates if candidate.exists()), None)
    if partition_path is None:
        raise FileNotFoundError(
            f"Не найдена партиция {source_name}/{partition_col}={partition_value}. "
            f"Проверенные варианты: {candidates}"
        )

    files = parquet_files(partition_path)
    if not files:
        raise FileNotFoundError(f"В партиции {partition_path} не найдены parquet-файлы")

    df = pl.read_parquet(files)
    df = normalize_source_columns(df)

    if partition_col not in df.columns:
        df = df.with_columns(pl.lit(str(partition_value)).alias(partition_col))
    else:
        df = df.with_columns(pl.col(partition_col).cast(pl.String))

    return df


def build_dim_sources():
    now = datetime.now()
    rows = []
    for source_name in ["deb_cards_info", "credit_cards_info", "client_cards_daily"]:
        cfg = SOURCE_CONFIG[source_name]
        rows.append({
            "source_id": cfg["source_id"],
            "source_name": cfg["source_name"],
            "source_description": cfg["source_description"],
            "update_frequency": cfg["update_frequency"],
            "row_create_dtime": now,
            "valid_to": "9999-12-31",
            "valid_from": "1900-01-01",
            "row_update_dtime": now,
        })
    return pl.DataFrame(rows, schema=DIM_SOURCES_SCHEMA)


def build_dim_attributes():
    now = datetime.now()
    rows = []
    attribute_id = 1

    for source_name in ["deb_cards_info", "credit_cards_info", "client_cards_daily"]:
        cfg = SOURCE_CONFIG[source_name]
        sample_df = read_source_partition(
            source_name,
            cfg["partition_col"],
            cfg["initial_partition"],
        )
        schema_by_name = {name: str(dtype) for name, dtype in sample_df.schema.items()}

        for column_name in cfg["business_columns"]:
            if column_name == "client_id" or column_name in TECHNICAL_COLUMNS:
                continue

            rows.append({
                "attribute_id": attribute_id,
                "attribute_name": column_name,
                "attribute_description": ATTRIBUTE_DESCRIPTIONS.get(column_name, f"Бизнес-атрибут {column_name}"),
                "data_type": schema_by_name.get(column_name, "String"),
                "source_id": cfg["source_id"],
                "update_frequency": cfg["update_frequency"],
                "row_create_dtime": now,
                "row_update_dtime": now,
            })
            attribute_id += 1

    return pl.DataFrame(rows, schema=DIM_ATTRIBUTES_SCHEMA)


def already_loaded(load_log_df, source_id, source_report_dt, target_table):
    return (
        load_log_df
        .filter(
            (pl.col("source_id") == int(source_id))
            & (pl.col("source_report_dt") == str(source_report_dt))
            & (pl.col("target_table") == target_table)
            & (pl.col("load_status") == "SUCCESS")
        )
        .height
        > 0
    )


def _next_load_id(load_log_df):
    if load_log_df.is_empty():
        return 1
    max_load_id = load_log_df.select(pl.max("load_id")).item()
    return 1 if max_load_id is None else int(max_load_id) + 1


def _source_loading_id(source_df):
    if "loading_id" not in source_df.columns or source_df.is_empty():
        return None
    value = source_df.select(pl.max("loading_id")).item()
    return None if value is None else int(value)


def add_load_log_record(
    load_log_df,
    source_id,
    source_report_dt,
    target_table,
    load_status="SUCCESS",
    loading_id=None,
    error_message=None,
    load_start_dtime=None,
):
    if load_start_dtime is None:
        load_start_dtime = datetime.now()
    load_end_dtime = datetime.now()

    new_df = pl.DataFrame([{
        "load_id": _next_load_id(load_log_df),
        "source_id": int(source_id),
        "source_report_dt": str(source_report_dt),
        "load_start_dtime": load_start_dtime,
        "load_end_dtime": load_end_dtime,
        "target_table": target_table,
        "load_status": load_status,
        "loading_id": loading_id,
        "error_message": error_message,
    }], schema=LOAD_LOG_SCHEMA)
    return pl.concat([load_log_df, new_df], how="vertical")


def _verticalize(source_df, source_name, dim_attributes_df, id_columns, output_columns):
    cfg = SOURCE_CONFIG[source_name]
    source_id = cfg["source_id"]
    attribute_columns = [c for c in cfg["business_columns"] if c != "client_id"]
    missing_columns = [c for c in attribute_columns if c not in source_df.columns]
    if missing_columns:
        raise ValueError(f"В источнике {source_name} нет колонок: {missing_columns}")

    prepared_df = source_df.with_columns(
        [pl.col("client_id").cast(pl.String)]
        + [pl.col(c).cast(pl.String).alias(c) for c in attribute_columns]
        + [pl.col(c).cast(pl.String).alias(c) for c in id_columns if c in source_df.columns and c != "client_id"]
        + ([pl.col("loading_id").cast(pl.Int64)] if "loading_id" in source_df.columns else [])
    )

    vertical_df = (
        prepared_df
        .unpivot(
            on=attribute_columns,
            index=id_columns + ["row_update_dtime", "loading_id", "row_hash_val"],
            variable_name="attribute_name",
            value_name="attribute_value",
        )
        .with_columns(pl.lit(source_id).cast(pl.Int32).alias("source_id"))
        .join(
            dim_attributes_df.select(["attribute_id", "attribute_name", "source_id"]),
            on=["attribute_name", "source_id"],
            how="left",
        )
        .with_columns(pl.col("attribute_id").cast(pl.Int32))
        .select(output_columns)
    )
    return vertical_df


def verticalize_monthly(source_df, source_name, dim_attributes_df):
    return _verticalize(
        source_df,
        source_name,
        dim_attributes_df,
        id_columns=["client_id", "report_dt"],
        output_columns=list(CLIENT_MONTHLY_SCHEMA.keys()),
    )


def verticalize_daily(source_df, source_name, dim_attributes_df):
    return _verticalize(
        source_df,
        source_name,
        dim_attributes_df,
        id_columns=["client_id", "row_actual_from", "row_actual_to"],
        output_columns=list(CLIENT_DAILY_SCHEMA.keys()),
    )


def _merge_by_latest(old_df, new_df, key_columns):
    if old_df.is_empty():
        return new_df.select(old_df.columns)
    if new_df.is_empty():
        return old_df

    sort_columns = key_columns + ["_is_new", "row_update_dtime", "loading_id"]
    descending = [False] * len(key_columns) + [True, True, True]

    merged = (
        pl.concat([
            old_df.with_columns(pl.lit(0).alias("_is_new")),
            new_df.with_columns(pl.lit(1).alias("_is_new")),
        ], how="vertical")
        .sort(sort_columns, descending=descending, nulls_last=True)
        .unique(subset=key_columns, keep="first", maintain_order=True)
        .drop("_is_new")
        .select(old_df.columns)
    )
    return merged


def merge_scd1(old_df, new_df):
    return _merge_by_latest(old_df, new_df, ["client_id", "attribute_id", "report_dt"])


def merge_scd2(old_df, new_df):
    return _merge_by_latest(old_df, new_df, ["client_id", "attribute_id", "row_actual_from"])


def check_duplicates(df, key_columns, table_name):
    duplicates = df.group_by(key_columns).len().filter(pl.col("len") > 1)
    print(f"Дубли в {table_name} по ключу {key_columns}: {duplicates.height}")
    print(duplicates)
    return duplicates


def show_table_info(df, table_name, n=10):
    print(f"\nТаблица: {table_name}")
    print(f"Количество строк: {df.height}")
    print(df.head(n))


# Секция 6. create_warehouse()

Функция создает пустое хранилище. Это первый шаг лабораторной: мы задаем физическую модель и записываем ее в parquet-таблицы.

В `dim_sources` сразу попадают три источника. В `dim_attributes` попадают только бизнес-атрибуты, кроме `client_id`, потому что `client_id` – это ключ клиента, а не измеряемый атрибут.

In [14]:
def create_warehouse():
    """Создает пять parquet-таблиц хранилища в директории warehouse_polars/."""
    Path(WAREHOUSE_DIR).mkdir(parents=True, exist_ok=True)

    dim_sources_df = build_dim_sources()
    dim_attributes_df = build_dim_attributes()
    load_log_df = empty_table("load_log")
    monthly_df = empty_table("client_monthly_attrs_scd1")
    daily_df = empty_table("client_daily_attrs_scd2")

    write_table(dim_sources_df, "dim_sources")
    write_table(dim_attributes_df, "dim_attributes")
    write_table(load_log_df, "load_log")
    write_table(monthly_df, "client_monthly_attrs_scd1")
    write_table(daily_df, "client_daily_attrs_scd2")

    show_table_info(dim_sources_df, "dim_sources")
    show_table_info(dim_attributes_df, "dim_attributes")
    print("Пустое Polars-хранилище создано.")


# Секция 7. initial_load_warehouse()

Первая загрузка моделирует начальное наполнение хранилища. Несмотря на то что parquet-файлы уже лежат статично в архиве, по смыслу лабораторной мы считаем, что это первая порция данных, пришедшая из источников.

Для месячных источников читаются партиции `report_dt='2023-02-28'`. Для дневного источника читается партиция `row_actual_to='2023-04-03'`. После чтения данные вертикализируются и записываются в целевые таблицы.

In [15]:
def initial_load_warehouse():
    """Выполняет первую загрузку начальных партиций в хранилище."""
    print("\n=== Первая загрузка хранилища ===")

    dim_attributes_df = read_table_if_exists("dim_attributes")
    load_log_df = read_table_if_exists("load_log")
    monthly_df = read_table_if_exists("client_monthly_attrs_scd1")
    daily_df = read_table_if_exists("client_daily_attrs_scd2")

    for source_name in MONTHLY_SOURCES:
        cfg = SOURCE_CONFIG[source_name]
        source_report_dt = cfg["initial_partition"]
        target_table = cfg["target_table"]

        if already_loaded(load_log_df, cfg["source_id"], source_report_dt, target_table):
            print(f"{source_name} за {source_report_dt} уже загружен. Пропускаю.")
            continue

        load_start = datetime.now()
        source_df = read_source_partition(source_name, cfg["partition_col"], source_report_dt)

        print(f"\nИсточник {source_name}, партиция {cfg['partition_col']}={source_report_dt}")
        print(source_df.schema)
        print(source_df.head(5))

        new_rows_df = verticalize_monthly(source_df, source_name, dim_attributes_df)
        print(f"Вертикализованные строки для {source_name}")
        print(new_rows_df.head(10))

        monthly_df = merge_scd1(monthly_df, new_rows_df)
        load_log_df = add_load_log_record(
            load_log_df,
            source_id=cfg["source_id"],
            source_report_dt=source_report_dt,
            target_table=target_table,
            loading_id=_source_loading_id(source_df),
            load_start_dtime=load_start,
        )

    cfg = SOURCE_CONFIG[DAILY_SOURCE]
    source_report_dt = cfg["initial_partition"]
    target_table = cfg["target_table"]

    if already_loaded(load_log_df, cfg["source_id"], source_report_dt, target_table):
        print(f"{DAILY_SOURCE} за {source_report_dt} уже загружен. Пропускаю.")
    else:
        load_start = datetime.now()
        source_df = read_source_partition(DAILY_SOURCE, cfg["partition_col"], source_report_dt)

        print(f"\nИсточник {DAILY_SOURCE}, партиция {cfg['partition_col']}={source_report_dt}")
        print(source_df.schema)
        print(source_df.head(5))

        new_rows_df = verticalize_daily(source_df, DAILY_SOURCE, dim_attributes_df)
        print(f"Вертикализованные строки для {DAILY_SOURCE}")
        print(new_rows_df.head(10))

        daily_df = merge_scd2(daily_df, new_rows_df)
        load_log_df = add_load_log_record(
            load_log_df,
            source_id=cfg["source_id"],
            source_report_dt=source_report_dt,
            target_table=target_table,
            loading_id=_source_loading_id(source_df),
            load_start_dtime=load_start,
        )

    write_table(monthly_df, "client_monthly_attrs_scd1")
    write_table(daily_df, "client_daily_attrs_scd2")
    write_table(load_log_df, "load_log")

    show_table_info(read_table_if_exists("client_monthly_attrs_scd1"), "client_monthly_attrs_scd1")
    show_table_info(read_table_if_exists("client_daily_attrs_scd2"), "client_daily_attrs_scd2")
    show_table_info(read_table_if_exists("load_log"), "load_log")


# Секция 8. update_warehouse()

Вторая загрузка моделирует обновление хранилища. Теперь читаются мартовские месячные партиции и актуальная дневная партиция `row_actual_to='9999-12-31'`.

Перед загрузкой каждой партиции функция смотрит в `load_log`. Если такая партиция уже была успешно загружена в эту целевую таблицу, повторная загрузка пропускается. Это простая защита от дублей.

In [16]:
def update_warehouse():
    """Выполняет вторую загрузку и не загружает уже обработанные партиции повторно."""
    print("\n=== Вторая загрузка хранилища ===")

    dim_attributes_df = read_table_if_exists("dim_attributes")
    load_log_df = read_table_if_exists("load_log")
    monthly_df = read_table_if_exists("client_monthly_attrs_scd1")
    daily_df = read_table_if_exists("client_daily_attrs_scd2")

    for source_name in MONTHLY_SOURCES:
        cfg = SOURCE_CONFIG[source_name]
        source_report_dt = cfg["update_partition"]
        target_table = cfg["target_table"]

        if already_loaded(load_log_df, cfg["source_id"], source_report_dt, target_table):
            print(f"{source_name} за {source_report_dt} уже загружен. Пропускаю.")
            continue

        load_start = datetime.now()
        source_df = read_source_partition(source_name, cfg["partition_col"], source_report_dt)

        print(f"\nИсточник {source_name}, партиция {cfg['partition_col']}={source_report_dt}")
        print(source_df.schema)
        print(source_df.head(5))

        new_rows_df = verticalize_monthly(source_df, source_name, dim_attributes_df)
        print(f"Вертикализованные строки для {source_name}")
        print(new_rows_df.head(10))

        monthly_df = merge_scd1(monthly_df, new_rows_df)
        load_log_df = add_load_log_record(
            load_log_df,
            source_id=cfg["source_id"],
            source_report_dt=source_report_dt,
            target_table=target_table,
            loading_id=_source_loading_id(source_df),
            load_start_dtime=load_start,
        )

    cfg = SOURCE_CONFIG[DAILY_SOURCE]
    source_report_dt = cfg["update_partition"]
    target_table = cfg["target_table"]

    if already_loaded(load_log_df, cfg["source_id"], source_report_dt, target_table):
        print(f"{DAILY_SOURCE} за {source_report_dt} уже загружен. Пропускаю.")
    else:
        load_start = datetime.now()
        source_df = read_source_partition(DAILY_SOURCE, cfg["partition_col"], source_report_dt)

        print(f"\nИсточник {DAILY_SOURCE}, партиция {cfg['partition_col']}={source_report_dt}")
        print(source_df.schema)
        print(source_df.head(5))

        new_rows_df = verticalize_daily(source_df, DAILY_SOURCE, dim_attributes_df)
        print(f"Вертикализованные строки для {DAILY_SOURCE}")
        print(new_rows_df.head(10))

        daily_df = merge_scd2(daily_df, new_rows_df)
        load_log_df = add_load_log_record(
            load_log_df,
            source_id=cfg["source_id"],
            source_report_dt=source_report_dt,
            target_table=target_table,
            loading_id=_source_loading_id(source_df),
            load_start_dtime=load_start,
        )

    write_table(monthly_df, "client_monthly_attrs_scd1")
    write_table(daily_df, "client_daily_attrs_scd2")
    write_table(load_log_df, "load_log")

    show_table_info(read_table_if_exists("client_monthly_attrs_scd1"), "client_monthly_attrs_scd1")
    show_table_info(read_table_if_exists("client_daily_attrs_scd2"), "client_daily_attrs_scd2")
    show_table_info(read_table_if_exists("load_log"), "load_log")


# Секция 9. Проверки результата

В конце нужны короткие проверки, которые удобно показать преподавателю. Мы смотрим количество строк, показываем содержимое таблиц, проверяем справочник источников, проверяем `attribute_id` после вертикализации и ищем дубли по ключам SCD1 и SCD2.

In [17]:
def run_simple_checks():
    """Показывает простые проверки результата лабораторной."""
    print("\n=== Проверки результата ===")

    dim_sources_df = read_table_if_exists("dim_sources")
    dim_attributes_df = read_table_if_exists("dim_attributes")
    load_log_df = read_table_if_exists("load_log")
    monthly_df = read_table_if_exists("client_monthly_attrs_scd1")
    daily_df = read_table_if_exists("client_daily_attrs_scd2")

    for table_name, df in [
        ("dim_sources", dim_sources_df),
        ("dim_attributes", dim_attributes_df),
        ("load_log", load_log_df),
        ("client_monthly_attrs_scd1", monthly_df),
        ("client_daily_attrs_scd2", daily_df),
    ]:
        show_table_info(df, table_name)

    print("\nПроверка, что все используемые source_id есть в dim_sources")
    source_ids_from_facts = pl.concat([
        dim_attributes_df.select("source_id"),
        load_log_df.select("source_id"),
        monthly_df.select("source_id"),
        daily_df.select("source_id"),
    ], how="vertical").unique()
    missing_source_ids = source_ids_from_facts.join(
        dim_sources_df.select("source_id").unique(),
        on="source_id",
        how="anti",
    )
    print(missing_source_ids)

    print("\nПроверка null в attribute_id после вертикализации")
    print(monthly_df.filter(pl.col("attribute_id").is_null()))
    print(daily_df.filter(pl.col("attribute_id").is_null()))

    check_duplicates(
        monthly_df,
        ["client_id", "attribute_id", "report_dt"],
        "client_monthly_attrs_scd1",
    )
    check_duplicates(
        daily_df,
        ["client_id", "attribute_id", "row_actual_from"],
        "client_daily_attrs_scd2",
    )

    print("\nЖурнал загрузок")
    print(load_log_df.sort("load_id"))


## Запуск всех шагов

Ниже главный сценарий лабораторной. Он специально короткий: все детали спрятаны в простые функции, а порядок действий хорошо читается сверху вниз.

In [18]:
# Полный линейный запуск лабораторной работы.
# Шаг 1 финального запуска: создаем физические parquet-таблицы и справочники.
create_warehouse()
# Шаг 2 финального запуска: загружаем начальные партиции источников.
initial_load_warehouse()
# Шаг 3 финального запуска: имитируем инкрементальное обновление новыми партициями.
update_warehouse()
# Шаг 4 финального запуска: выводим контрольные проверки результата.
run_simple_checks()



Таблица: dim_sources
Количество строк: 3
shape: (3, 8)
┌───────────┬────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┐
│ source_id ┆ source_nam ┆ source_des ┆ update_fre ┆ row_creat ┆ valid_to  ┆ valid_fro ┆ row_updat │
│ ---       ┆ e          ┆ cription   ┆ quency     ┆ e_dtime   ┆ ---       ┆ m         ┆ e_dtime   │
│ i32       ┆ ---        ┆ ---        ┆ ---        ┆ ---       ┆ str       ┆ ---       ┆ ---       │
│           ┆ str        ┆ str        ┆ str        ┆ datetime[ ┆           ┆ str       ┆ datetime[ │
│           ┆            ┆            ┆            ┆ μs]       ┆           ┆           ┆ μs]       │
╞═══════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 1         ┆ deb_cards_ ┆ Данные о   ┆ 1 месяц    ┆ 2026-05-2 ┆ 9999-12-3 ┆ 1900-01-0 ┆ 2026-05-2 │
│           ┆ info       ┆ транзакцио ┆            ┆ 1 20:30:0 ┆ 1         ┆ 1         ┆ 1 20:30:0 │
│           ┆            ┆ нной    

# Секция 10. Краткое объяснение результата

После запуска в папке `warehouse_polars/` появляются пять parquet-таблиц. Это и есть учебное хранилище данных.

`dim_sources` отвечает на вопрос, откуда пришли данные. `dim_attributes` отвечает на вопрос, какие бизнес-атрибуты клиента мы храним. `load_log` отвечает на вопрос, какие партиции уже загружены. `client_monthly_attrs_scd1` хранит месячные атрибуты в вертикальном виде. `client_daily_attrs_scd2` хранит дневные атрибуты в вертикальном виде и сохраняет период действия строк.

Основное допущение: сами исходные parquet-файлы статичны, но в лабораторной мы моделируем процесс обновления так, как будто сначала пришли начальные партиции, а затем пришли новые партиции.